INTEGRAZIONE MODELLI MULTIPLI

I problemi raramente sono confinati in un unico contesto.
Immagina un robot magazziniere, non deve solo spostare un pacco, deve vedere il paccco (CV), leggere l'etichetta (OCR), capire la destinazione (NLP), decidere dove portarlo e portarlo (logica/azione)
Un singolo modello non può fare tutto. Serve una catena dove ogni anella è specializzato.
Modelli di natura diversa devono essere collegati per creare una soluzione che supera le capacità di un singolo modello

- Pipeline a catena: come far comunicare computer vision e modelli linguistici
- Ensembling: strategie di aggregazione per la robustezza delle predizioni
- Gestione fallimenti: pattern di fallback e circuit breaker per catene di modelli.

Il passagio del testimone tra pixel e parole
Pensa ad un'applicazione di gestione spese. L'utente scatta la foto allo scontrino, un singolo modello non può gestirlo in modo totale. Abbiamo bisogno di una staffetta. Il primo corridore è la computer vision, che estrae il testo grezzo dai pixel. Il secondo corridere prende quel testo e lo trasforma in una  riga di contabilità (LLM).
Se l'OCR legge male una cifra, come può il modello linguistico capire l'errore?
La sfida è tecnica, come passiamo quei dati dal primo corridore al secondo, evitando la perdita di informazioni spaziali o contestuali.

Architettura della Catena
Coordinare visione e linguaggio
Le tre modalità di comunicazione
* OCR to LLM (la più comune): utilizzo di modelli come Tesseract o EasyOCR per convertire pixel in stringhe, passi la stringa all'LLM con un prompt che dice: "correggi eventuali errori OCR"
* Feature injection (più sofisticata): passaggio di embedding visuali (estratti da una CNN o ViT) direttamente nello spazio latente di un Transformer linguistico. Non perdi informazioni spaziali. Il costo? i due modelli devono essere progettati per parlarsi a livello matematico
* Modularity: possibilità di aggiornare il 'motore di visione' senza dover riaddestrare la logica di business contenuta nel modello NLP
* L'inferenza di sistema è definita come la composizione di funzioni applicate sequenzialmente all'input originale.

Come ottimiziamo questo passaggio affinchè sia fluido e non generi errori

Non basta collegare i cavi, serve condizionare i modelli
Il Prompt Conditioning è fondamentale, diciamo all'LLM: guarda riceverai del testo da un OCR, potrebbe contenere errori come uno 0 (numero) al posto di una o (testo), correggilo.
I modelli collaborano in tempo reale, per rispondere a domande come: "che colore è la maglietta nella foto?"
Impostare tempi massimi di attesa brevi per i modelli secondari per non compromettere la latenza totale della pipeline.
Infine usiamo la Data Serialization, i modelli vivono in container docker separati e comunicano tramite JSON, è la burocrazia necessaria affichè due specialisti diversi si capiscano facilmente.

Ma attenzione, più la catena è lunga più è facile inciampare.
Propagazione dell'Errore
Il rischio di catene lunghe.
In una pipeline sequenziale, l'errore del sistema finale è legato alla somma degli errori di ogni singolo stadio.
La probabilità di successo del sistema diminuisce esponenzialmente all'aumentare dei modelli collegati senza meccanismi di correzione intermedi.
E' il gioco dei telefono senza fili.

Per questo dobbiamo inserire meccanismi di controllo e validazione tra uno stadio e l'altro 

Ensembling per la Produzione
La saggezza della folla algoritmica
Invece di affidarci a un unico modello 'campione', possiamo interrogare simultaneamente più modelli diversi e mediare le loro risposta.
L'ensembling riduce la varianza delle predizioni e protegge il sistema dai 'bias' specifici di una singola archirettura o di un particolare dateset di training.
Se 3 modelli diversi addestrati su architetture diverse, concordano nella risposta, la nostra fiducia aumenta.


Quali sono le tecniche matematiche per unire questi pareri.

Tecniche di Aggregazione
- il Majorit Voting è la democrazia pura: scelta della classe predetta dalla maggioranza dei modelli (utile per classificazione pure)
- Weighted Averaging: assegnazione di un peso maggiore al modello più affidabile (es. SOTA) mediando le probabilità logaritmiche.
- Stacking: utilizzo di un tezo modello (meta-learner) che impara a combinare gli output dei primi due per ottimizzare la precisione. 
 La predizione finale è la media pesata delle distribuzioni di probabilità fornite dai singoli modeli dell'ensemble.

Ovviamente tutto questo ha un costo che dobbiamo saper gestire.
Più modelli + Più precisione - velocità
Se ogni modello impiega 100ms, usarne 3 significa almeno 300ms (se girano il sequenza) o poco meno (se in parallelo). In alcuni contesti questo è accettabile, in altri assolutamente no.
Dove vale la pena:
- Diagnosi mediche: meglio aspettare 2 secondi in più per una risposta affidabile
- Transazioni finanziarie: un falso positivo può costare milioni
Dove non conviene:
- Suggerire il prossimo video tiktok
- Ricerca in tempo reale, come google
L'abilità sta nel staper scegliere quando usare un singolo modello veloce e quando vale la pena pagare il costo extra di latenza per avere più robustezza.

Diversità dei Modelli
- Architetture Diversity: combinare un Transformer con una CNN è utilizzimo, perchè vedono il mondo in modo strutturalmente diversi, tendono a fare errori diversi.
- Cross-Validation Ensembling: utilizzare le versioni del modello salvate durante diverse fold della cross-validation per stabilizzare l'inferenza
- Inference Latency Cost: valutazione del compromesso: l'ensemble aumenta la precisione ma raddoppia o triplica il tempo di costo di calcolo

La matematica ci spiega perchè l'unione di più modelli funziona quasi sempre
L'errore atteso di un ensemble è sistematicamente inferiore o uguale all'errore medio dei suoi componenti se i modelli sono decorrelati.
Il guadagno prestazionale è proporzionale all'ambiguità (diversità) tra le risposte dei singoli modelli selezionati.
Bilanciamento tra bias e varianza, significa che più i modelli sono diversi tra loro, più l'errore cala, se usate 3 modelli identici, l'errore resta lo stesso.

Ma cosa succede quando uno dei modelli si rompe o non è sicuro?
Parliamo di resilienza
Resilienza e Gestione Fallimenti.
Costruire sistemi AI che non crollano
In una pipeline complessa, un timeout di rete o un errore di memoria in un modello non deve abbattere l'intera applicazione.
In produzione i server cadono, le GPU finiscono la memoria ed i dati arrivano corrotti, non possiamo permettere che un errore in un sottosistema trascini nel baratro l'intera esperienza utente.
Implemetare logiche di 'graceful degradation' per garantire che l'utente riceva sempre una risposta, anche se parziale o semplificata.
Se l'ai non funziona, il sistema deve trasformarsi in un software tradizionale più semplice, ma comunque funzionante.

Graceful Degradation
Il concetto chiave: se l'AI non può rispondere, il sistema non deve dare errore 500, ma deve saper fare un passo indietro.
Immagina un assistente vocale per un magazzino. Se il modello di visione non funziona, invece di crashare, il sistema potrebbe chidere all'operatore di digitare manualmente il codice del pacco. E' più lento, ma funziona comunque.

La nostra prima linea di difesa è la consapevolezza dei limiti

Stregegie di Fallback
Progettare per l'errore
- Confidence Thresholding: prima linea di difesa: se il modello principale ci dice 'sono sicuro al 40%' non dobbiamo mostrare quella risposta all'utente ma dobbiamo attivare automaticamente un modello di bakcup più piccolo ma più robusto e conservativo. (consiglio attivare modello backup sotto il 60%)
- Oppure possiamo dare anche una risposta statica predefinita, o un messaggio di errore 'umano' se l'intera catena di modelli fallisce.
- Oppure possiamo chiedere all'utente (human in the loop)
- Dobbiamo gestire il Try-Except for AI: pattern di programmazione per catturare eccezioni hardware (es. out of memory) e reindirizzare la richiesta su una CPU per non interrompere il servizio.

Per evitare che un guasto mandi in crash l'intero sistema, prendiamo un concetto in prestito dall'elettronica
- Circuit Breaker: come l'interrutore di casa, se un modello inizia a sbagliare a raffica, smetti di chiamarlo per un pò. Gli da il tempo di riprendersi, invece di continuare a inondarlo di richieste che falliranno.
- Timeouts Stratificati: impostare tempi massimi di attesa brevi per i modelli secondari per non compromettere la latenza totale della pipeline.
- Human-in-the-loop: pattern in cui i casi falliti dall'AI vengono accodati per una revisione manuale da parte di un operatore umano. 
L'AI migliore è quella che sa quando fermarsi

Disponibilità del Sistema
Metriche di affidabilità
L'integrazione di modelli multipli deve puntare alla massimizzazione dell'uptime del servizio percepito dall'utente
L'affidabilità totale è calcolata come il tempo medio tra i guasti (MTBF) rapportato al tempo toale di operatività richiesto.
Un sistema con molti modelli integrati ha più punti di rottura, ma se progettato con il fallback, che abbiamo visto, la disponibilità percepita dall'utente sarà molto più alta di quelli dei singoli componenti.
Vogliamo che il servizio non dica mai errore 500 ma che trovi una strada alternativa per servire l'utente.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- COMPONENTI DELLA ARCHITETTURA ---

class VisionEncoder(nn.Module):
    """
    Questa classe si occupa di estrarre le caratteristiche salienti (feature) da una immagine.
    Rappresenta il primo stadio della pipeline, trasformando i pixel grezzi in un vettore numerico.
    """
    def __init__(self):
        super().__init__()
        # Definiamo la sequenza di trasformazioni per processare l'immagine
        self.features = nn.Sequential(
            # Applica una convoluzione per estrarre pattern spaziali (bordi, forme)
            nn.Conv2d(3, 16, kernel_size=3), #(ricevi 3 canali, produci 16 canali, usa filtri con finestra spaziale 3x3)
            # Riduce le dimensioni spaziali a 1x1, mantenendo solo l'essenza delle feature
            nn.AdaptiveAvgPool2d((1, 1)),
            # Trasforma il tensore multidimensionale in un vettore piatto (vettore di feature)
            nn.Flatten()
        )
    
    def forward(self, x):
        # Riceve l'immagine in input e restituisce il vettore di feature estratto
        return self.features(x)

class TextClassifier(nn.Module):
    """
    Questa classe funge da classificatore. Riceve in input le feature estratte dal VisionEncoder
    e le utilizza per determinare la categoria di appartenenza (es. Positivo/Negativo).
    """
    def __init__(self):
        super().__init__()
        # Trasforma il vettore di 16 feature in 2 valori (logits) corrispondenti alle classi di output
        self.classifier = nn.Linear(16, 2)
    
    def forward(self, x):
        # Calcola i punteggi (logits) per ogni classe partendo dalle feature ricevute
        return self.classifier(x)

# --- GESTORE CENTRALIZZATO (ORCHESTRATORE) ---

class RobustSystem:
    """
    Questa classe orchestra l'intero processo: coordina il modello di visione,
    gestisce un ensemble di classificatori e applica logiche di sicurezza (fallback).
    """
    def __init__(self, vision_model, nlp_models, confidence_threshold=0.8):
        # Il modello che estrae le feature dalle immagini
        self.vision = vision_model
        # Una lista di modelli che lavorano insieme per migliorare la precisione (Ensemble)
        self.nlp_ensemble = nn.ModuleList(nlp_models)
        # Livello minimo di confidenza richiesto per accettare una previsione automatica
        self.threshold = confidence_threshold
        # Valore restituito se il sistema non è abbastanza sicuro della previsione
        self.fallback_value = "RICHIESTA_REVISIONE_UMANA"

    def predict(self, image_tensor):
        """
        Esegue l'intero flusso di lavoro: Visione -> Integrazione -> Decisione.
        """
        # Disabilita il calcolo del gradiente per risparmiare memoria durante l'inferenza
        with torch.inference_mode():
            
            # FASE 1: Estrazione delle caratteristiche visive tramite il VisionEncoder
            # Il sistema passa l'immagine al componente di visione per ottenere il vettore di feature
            visual_features = self.vision(image_tensor)
            
            # FASE 2: Classificazione tramite Ensemble (Media Pesata)
            # Interroghiamo tutti i modelli presenti nella lista per ottenere più "opinioni"
            all_probs = []
            for model in self.nlp_ensemble:
                # Ogni modello riceve le stesse feature visive estratte in precedenza
                logits = model(visual_features)
                # Trasformiamo i punteggi grezzi in probabilità (che sommano a 1)
                probs = F.softmax(logits, dim=1)
                all_probs.append(probs)
            
            # Calcoliamo la media delle probabilità fornite da tutti i modelli dell'ensemble
            # Questo riduce il rischio che un singolo errore di un modello comprometta il risultato
            avg_probs = torch.stack(all_probs).mean(dim=0)
            
            # Identifichiamo la classe con la probabilità più alta e il relativo valore di confidenza
            conf, pred = torch.max(avg_probs, dim=1)
            
            # FASE 3: Logica di Controllo Qualità (Fallback)
            # Se la confidenza è inferiore alla soglia impostata, attiviamo il fallback umano
            if conf.item() < self.threshold:
                return self.fallback_value, conf.item()
            
            # Altrimenti restituiamo la previsione del sistema
            return pred.item(), conf.item()

# --- ESEMPIO DI ESECUZIONE ---

# 1. Creazione dei componenti fondamentali
modello_visione = VisionEncoder()

# 2. Preparazione dell'ensemble: utilizziamo più istanze del classificatore per maggiore robustezza
# Qui creiamo una lista di due modelli indipendenti che verranno consultati dal sistema
lista_modelli_classificazione = [TextClassifier(), TextClassifier()]

# 3. Inizializzazione del sistema robusto con una soglia di attendibilità del 75%
# L'orchestratore riceve il vision encoder e la lista dei classificatori
sistema = RobustSystem(modello_visione, lista_modelli_classificazione, confidence_threshold=0.75)

# 4. Generazione di un dato di esempio (simula un'immagine a colori di 32x32 pixel)
# La forma [1, 3, 32, 32] rappresenta: [batch_size, canali_RGB, altezza, larghezza]
immagine_esempio = torch.randn(1, 3, 32, 32)

# 5. Esecuzione della previsione attraverso l'intero sistema orchestrato
# La funzione predict gestisce internamente il passaggio di dati tra visione e classificazione
risultato, punteggio_confidenza = sistema.predict(immagine_esempio)

# 6. Visualizzazione dell'output finale
# Mostriamo se il sistema ha deciso in autonomia o se è necessario l'intervento umano
print(f"Esito del sistema: {risultato} (Livello di confidenza: {punteggio_confidenza:.2f})")

Esito del sistema: RICHIESTA_REVISIONE_UMANA (Livello di confidenza: 0.57)
